# LightGBM — Fake News Classifier

**Task:** Binary classification of political statements as true (0) or false (1).

**Why LightGBM?** After Random Forest, LightGBM brought two key improvements:
- **Leaf-wise tree growth**: LightGBM grows the leaf with the largest loss reduction at each step (best-first), rather than level-wise like RFC. For the same number of leaves this produces deeper, more expressive trees — faster and often more accurate.
- **Explicit regularisation**: `reg_alpha` (L1) and `reg_lambda` (L2) directly penalise leaf weights, giving finer control over overfitting than RFC's `min_samples_leaf` alone.

**The speaker true-rate discovery:** LightGBM's gain-based feature importance revealed that `fe_speaker_true_rate` captured **6–8× more gain** than the next feature. This led to three experiments:
- **Option A**: all true-rate features included (baseline)
- **Option B**: drop `fe_speaker_true_rate` — forced the model to use subject/party/job rates and embeddings
- **Option C** *(this notebook)*: restore `fe_speaker_true_rate` + add L1/L2 regularisation (`reg_alpha`, `reg_lambda`) to reduce its dominance without removing the signal

**Architecture:**
- Sentence embeddings (`all-MiniLM-L6-v2`, 384-dim) for the statement text
- Rich metadata with interaction keys and OrdinalEncoded categoricals
- All four true-rate features computed fold-safe inside the CV loop
- Nested CV: 5-fold outer, 3-fold inner `RandomizedSearchCV` (20 trials per fold)
- Threshold tuning on OOF probabilities, optimising macro F1

**Metric:** Macro F1 (primary) — forces the model to predict both classes.

## Imports and Project Setup

In [ ]:
from collections import Counter
from datetime import datetime
from pathlib import Path
import sys
from time import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'train.csv').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

project_root = find_project_root(Path.cwd())
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing.one_step import OneStepOptions, preprocess_one_step

print(f'Project root: {project_root}')

_script_start = time()
def _now() -> str:
    return datetime.now().strftime('%H:%M:%S')

## Data Loading

8,950 labeled training samples. Class distribution: ~35% true (0), ~65% false (1). The imbalance is handled via `class_weight={0: 1.42, 1: 0.77}` — computed as `n_samples / (2 * n_class_i)`. LightGBM scales the leaf weights during boosting, so class 0 samples contribute proportionally more to the loss.

In [ ]:
df = pd.read_csv(project_root / 'data' / 'train.csv')
print(f'Shape: {df.shape}')
print(f"\nLabel distribution:\n{df['label'].value_counts(normalize=True).round(4)}")
df.head()

## Preprocessing Configuration

All preprocessing is controlled through `OneStepOptions`. The LGBM configuration is identical to RFC — both are tree-based and share the same preprocessing decisions:

| Setting | Value | Reason |
|---|---|---|
| Scaling | `none` | Trees are invariant to monotone transforms |
| Stemmer | `none` | Trees may use token distinctions that stemming erases |
| Stopword removal | OFF | Sentence embeddings need natural language |
| NER features | ON | Entity counts (PERSON, ORG, GPE, DATE) useful for fact-checking |
| Interaction keys | ON | LightGBM splits directly on joint categorical keys |
| Embeddings | `all-MiniLM-L6-v2` | 384-dim semantic vectors |
| `fe_proper_noun_count` | OFF | `statement_original` is dropped; casing heuristic can't run |

In [ ]:
# --- Label & ID ---
label_option      = 'skip'
label_source_col  = 'label'
id_option         = 'drop'

# --- Subject ---
subject_source_col             = 'subject'
subject_keep_original          = False
subject_clean_text             = True
subject_normalize_separators   = True
subject_split_topics           = True
subject_primary_strategy       = 'most_frequent'
subject_rare_threshold         = 10
subject_rare_label             = 'other'
subject_max_topics_for_primary = None
subject_multi_topic_label      = 'multi-topic'
subject_add_primary            = True
subject_add_topic_count        = True
subject_add_multiple_topics_flag = True
subject_add_topic_list         = False
subject_add_length_features    = True
subject_add_grouped_primary    = True
subject_group_rare             = True
subject_add_subject_frequency  = True
subject_add_subject_is_rare    = True
subject_add_subject_primary_true_rate = False  # computed fold-safe in CV loop
subject_label_col              = None
subject_scale                  = 'none'
subject_verbose                = False

In [ ]:
# --- Statement ---
statement_source_col             = 'statement'
statement_original_output_col    = 'statement_original'
statement_output_col             = 'statement_clean'
statement_keep_original          = False
statement_lower                  = True
statement_remove_html            = True
statement_remove_urls            = True
statement_replace_numbers        = False
statement_number_token           = '<NUM>'
statement_stopword_removal       = False   # OFF: embeddings need natural language
statement_keep_negations         = True
statement_remove_punctuation     = False
statement_stemmer                = 'none'  # OFF: trees gain nothing from stemming
statement_lemmatizer             = 'none'
statement_repair_polluted_statements = True
statement_add_rare_token_features   = True
statement_rare_token_threshold      = 1
statement_token_freqs               = None
statement_add_spelling_errors       = True
statement_add_lexical_features      = True
statement_add_pollution_features    = True
statement_add_ner_features          = True
statement_ner_model                 = 'en_core_web_sm'
statement_vectorizer_type           = 'embeddings'
statement_vectorizer_max_features   = 500
statement_vectorizer_min_df         = 2
statement_vectorizer_max_df         = 0.7
statement_embedding_model           = 'all-MiniLM-L6-v2'
statement_fitted_vectorizer         = None
statement_scale                     = 'none'
statement_verbose                   = False

In [ ]:
# --- Speaker ---
speaker_source_col            = 'speaker'
speaker_keep_original         = False
speaker_clean_text            = True
speaker_normalize_separators  = True
speaker_group_rare            = True
speaker_rare_threshold        = 5
speaker_rare_label            = 'other'
speaker_add_frequency         = True
speaker_add_is_rare           = True
speaker_add_grouped_speaker   = True
speaker_add_length_features   = True
speaker_add_title_flag        = True
speaker_add_comma_flag        = True
speaker_add_period_flag       = True
speaker_add_token_count       = False
speaker_add_speaker_primary_true_rate = False
speaker_label_col             = None
speaker_scale                 = 'none'
speaker_verbose               = False

# --- Speaker job ---
speaker_job_source_col           = 'speaker_job'
speaker_job_keep_original        = False
speaker_job_clean_text           = True
speaker_job_normalize_separators = True
speaker_job_group_rare           = True
speaker_job_rare_threshold       = 5
speaker_job_rare_label           = 'other'
speaker_job_add_frequency        = True
speaker_job_add_is_rare          = True
speaker_job_add_grouped_job      = True
speaker_job_add_length_features  = True
speaker_job_add_title_flag       = True
speaker_job_add_comma_flag       = True
speaker_job_add_slash_flag       = True
speaker_job_add_ampersand_flag   = True
speaker_job_add_token_count      = False
speaker_job_add_job_primary_true_rate = False
speaker_job_job_label_col        = None
speaker_job_scale                = 'none'
speaker_job_verbose              = False

# --- Party affiliation ---
party_affiliation_source_col         = 'party_affiliation'
party_affiliation_keep_original      = False
party_affiliation_clean_text         = True
party_affiliation_group_rare         = True
party_affiliation_rare_threshold     = 5
party_affiliation_rare_label         = 'other'
party_affiliation_add_frequency      = True
party_affiliation_add_is_rare        = True
party_affiliation_add_grouped_party  = True
party_affiliation_add_length_features = True
party_affiliation_add_token_count    = True
party_affiliation_add_is_major_party = True
party_affiliation_add_is_institutional = True
party_affiliation_add_slash_flag     = True
party_affiliation_add_ampersand_flag = True
party_affiliation_add_comma_flag     = False
party_affiliation_add_parentheses_flag = True
party_affiliation_add_party_primary_true_rate = False
party_affiliation_party_label_col    = None
party_affiliation_scale              = 'none'
party_affiliation_verbose            = False

# --- State info ---
state_source_col        = 'state_info'
state_drop              = False
state_keep_original     = False
state_clean_text        = True
state_normalize_state   = True
state_group_rare        = True
state_rare_threshold    = 5
state_rare_label        = 'other'
state_add_is_us_state   = True
state_add_frequency     = True
state_add_is_rare       = True
state_add_grouped_state = True
state_add_has_us_words  = True
state_add_us_region     = True
state_add_length_features = False
state_add_token_count   = True
state_scale             = 'none'
state_verbose           = False

In [ ]:
# --- Feature Engineering ---
fe_statement_col          = 'statement_clean'
fe_statement_original_col = 'statement_original'
fe_speaker_col            = 'speaker_clean'
fe_subject_col            = 'subject_clean'
fe_party_col              = 'party_affiliation_clean'
fe_speaker_job_col        = 'speaker_job_clean'
fe_state_col              = 'state_info_clean'
fe_label_col              = None  # true-rates computed fold-safe in CV loop

# 1. Interaction features (joint string keys → OrdinalEncoded)
fe_add_speaker_subject              = True
fe_add_speaker_party                = True
fe_add_subject_party                = True
fe_add_speaker_job_subject          = True
fe_add_state_party                  = True
fe_add_speaker_statement_len_bucket = True
fe_statement_len_bins               = (50, 150)

# 2. True-rate aggregates — False here, computed fold-safe inside CV loop
fe_add_speaker_true_rate  = False
fe_add_subject_true_rate  = False
fe_add_party_true_rate    = False

# 2b. Non-leaking aggregates
fe_add_speaker_avg_statement_len = True
fe_add_subject_avg_statement_len = True
fe_add_speaker_avg_punctuation   = True
fe_add_speaker_avg_number_ratio  = True

# 3. Text-style features
fe_add_negation_count    = True
fe_add_hedge_count       = True
fe_add_absolutist_count  = True
fe_add_numeral_count     = True
fe_add_proper_noun_count = False  # statement_original is dropped; casing heuristic can't run
fe_add_readability       = True
fe_add_sentiment         = True

fe_scale    = 'none'
fe_verbose  = False

## Build OneStepOptions

In [ ]:
options = OneStepOptions(
    label_option=label_option, label_source_col=label_source_col, id_option=id_option,
    subject_source_col=subject_source_col, subject_keep_original=subject_keep_original,
    subject_clean_text=subject_clean_text, subject_normalize_separators=subject_normalize_separators,
    subject_split_topics=subject_split_topics, subject_primary_strategy=subject_primary_strategy,
    subject_max_topics_for_primary=subject_max_topics_for_primary,
    subject_multi_topic_label=subject_multi_topic_label,
    subject_add_primary=subject_add_primary, subject_add_topic_count=subject_add_topic_count,
    subject_add_multiple_topics_flag=subject_add_multiple_topics_flag,
    subject_add_topic_list=subject_add_topic_list, subject_add_length_features=subject_add_length_features,
    subject_add_grouped_primary=subject_add_grouped_primary, subject_group_rare=subject_group_rare,
    subject_rare_threshold=subject_rare_threshold, subject_rare_label=subject_rare_label,
    subject_add_subject_frequency=subject_add_subject_frequency,
    subject_add_subject_is_rare=subject_add_subject_is_rare,
    subject_add_subject_primary_true_rate=subject_add_subject_primary_true_rate,
    subject_label_col=subject_label_col, subject_scale=subject_scale, subject_verbose=subject_verbose,
    statement_source_col=statement_source_col, statement_original_output_col=statement_original_output_col,
    statement_output_col=statement_output_col, statement_keep_original=statement_keep_original,
    statement_lower=statement_lower, statement_remove_html=statement_remove_html,
    statement_remove_urls=statement_remove_urls, statement_replace_numbers=statement_replace_numbers,
    statement_number_token=statement_number_token, statement_stopword_removal=statement_stopword_removal,
    statement_keep_negations=statement_keep_negations, statement_stemmer=statement_stemmer,
    statement_lemmatizer=statement_lemmatizer, statement_remove_punctuation=statement_remove_punctuation,
    statement_repair_polluted_statements=statement_repair_polluted_statements,
    statement_add_rare_token_features=statement_add_rare_token_features,
    statement_rare_token_threshold=statement_rare_token_threshold,
    statement_token_freqs=statement_token_freqs, statement_add_spelling_errors=statement_add_spelling_errors,
    statement_add_lexical_features=statement_add_lexical_features,
    statement_add_pollution_features=statement_add_pollution_features,
    statement_add_ner_features=statement_add_ner_features, statement_ner_model=statement_ner_model,
    statement_vectorizer_type=statement_vectorizer_type,
    statement_vectorizer_max_features=statement_vectorizer_max_features,
    statement_vectorizer_min_df=statement_vectorizer_min_df,
    statement_vectorizer_max_df=statement_vectorizer_max_df,
    statement_embedding_model=statement_embedding_model,
    statement_fitted_vectorizer=statement_fitted_vectorizer,
    statement_scale=statement_scale, statement_verbose=statement_verbose,
    speaker_source_col=speaker_source_col, speaker_keep_original=speaker_keep_original,
    speaker_clean_text=speaker_clean_text, speaker_normalize_separators=speaker_normalize_separators,
    speaker_group_rare=speaker_group_rare, speaker_rare_threshold=speaker_rare_threshold,
    speaker_rare_label=speaker_rare_label, speaker_add_frequency=speaker_add_frequency,
    speaker_add_is_rare=speaker_add_is_rare, speaker_add_grouped_speaker=speaker_add_grouped_speaker,
    speaker_add_length_features=speaker_add_length_features, speaker_add_title_flag=speaker_add_title_flag,
    speaker_add_comma_flag=speaker_add_comma_flag, speaker_add_period_flag=speaker_add_period_flag,
    speaker_add_token_count=speaker_add_token_count,
    speaker_add_speaker_primary_true_rate=speaker_add_speaker_primary_true_rate,
    speaker_label_col=speaker_label_col, speaker_scale=speaker_scale, speaker_verbose=speaker_verbose,
    speaker_job_source_col=speaker_job_source_col, speaker_job_keep_original=speaker_job_keep_original,
    speaker_job_clean_text=speaker_job_clean_text,
    speaker_job_normalize_separators=speaker_job_normalize_separators,
    speaker_job_group_rare=speaker_job_group_rare, speaker_job_rare_threshold=speaker_job_rare_threshold,
    speaker_job_rare_label=speaker_job_rare_label, speaker_job_add_frequency=speaker_job_add_frequency,
    speaker_job_add_is_rare=speaker_job_add_is_rare, speaker_job_add_grouped_job=speaker_job_add_grouped_job,
    speaker_job_add_length_features=speaker_job_add_length_features,
    speaker_job_add_title_flag=speaker_job_add_title_flag,
    speaker_job_add_comma_flag=speaker_job_add_comma_flag, speaker_job_add_slash_flag=speaker_job_add_slash_flag,
    speaker_job_add_ampersand_flag=speaker_job_add_ampersand_flag,
    speaker_job_add_token_count=speaker_job_add_token_count,
    speaker_job_add_job_primary_true_rate=speaker_job_add_job_primary_true_rate,
    speaker_job_job_label_col=speaker_job_job_label_col,
    speaker_job_scale=speaker_job_scale, speaker_job_verbose=speaker_job_verbose,
    party_affiliation_source_col=party_affiliation_source_col,
    party_affiliation_keep_original=party_affiliation_keep_original,
    party_affiliation_clean_text=party_affiliation_clean_text,
    party_affiliation_group_rare=party_affiliation_group_rare,
    party_affiliation_rare_threshold=party_affiliation_rare_threshold,
    party_affiliation_rare_label=party_affiliation_rare_label,
    party_affiliation_add_frequency=party_affiliation_add_frequency,
    party_affiliation_add_is_rare=party_affiliation_add_is_rare,
    party_affiliation_add_grouped_party=party_affiliation_add_grouped_party,
    party_affiliation_add_length_features=party_affiliation_add_length_features,
    party_affiliation_add_token_count=party_affiliation_add_token_count,
    party_affiliation_add_is_major_party=party_affiliation_add_is_major_party,
    party_affiliation_add_is_institutional=party_affiliation_add_is_institutional,
    party_affiliation_add_slash_flag=party_affiliation_add_slash_flag,
    party_affiliation_add_ampersand_flag=party_affiliation_add_ampersand_flag,
    party_affiliation_add_comma_flag=party_affiliation_add_comma_flag,
    party_affiliation_add_parentheses_flag=party_affiliation_add_parentheses_flag,
    party_affiliation_add_party_primary_true_rate=party_affiliation_add_party_primary_true_rate,
    party_affiliation_party_label_col=party_affiliation_party_label_col,
    party_affiliation_scale=party_affiliation_scale, party_affiliation_verbose=party_affiliation_verbose,
    state_source_col=state_source_col, state_drop=state_drop, state_keep_original=state_keep_original,
    state_clean_text=state_clean_text, state_normalize_state=state_normalize_state,
    state_group_rare=state_group_rare, state_rare_threshold=state_rare_threshold,
    state_rare_label=state_rare_label, state_add_is_us_state=state_add_is_us_state,
    state_add_frequency=state_add_frequency, state_add_is_rare=state_add_is_rare,
    state_add_grouped_state=state_add_grouped_state, state_add_length_features=state_add_length_features,
    state_add_token_count=state_add_token_count, state_add_has_us_words=state_add_has_us_words,
    state_add_us_region=state_add_us_region, state_scale=state_scale, state_verbose=state_verbose,
    fe_statement_col=fe_statement_col, fe_statement_original_col=fe_statement_original_col,
    fe_speaker_col=fe_speaker_col, fe_subject_col=fe_subject_col, fe_party_col=fe_party_col,
    fe_speaker_job_col=fe_speaker_job_col, fe_state_col=fe_state_col, fe_label_col=fe_label_col,
    fe_add_speaker_subject=fe_add_speaker_subject, fe_add_speaker_party=fe_add_speaker_party,
    fe_add_subject_party=fe_add_subject_party, fe_add_speaker_job_subject=fe_add_speaker_job_subject,
    fe_add_state_party=fe_add_state_party,
    fe_add_speaker_statement_len_bucket=fe_add_speaker_statement_len_bucket,
    fe_statement_len_bins=fe_statement_len_bins,
    fe_add_speaker_true_rate=fe_add_speaker_true_rate, fe_add_subject_true_rate=fe_add_subject_true_rate,
    fe_add_party_true_rate=fe_add_party_true_rate,
    fe_add_speaker_avg_statement_len=fe_add_speaker_avg_statement_len,
    fe_add_subject_avg_statement_len=fe_add_subject_avg_statement_len,
    fe_add_speaker_avg_punctuation=fe_add_speaker_avg_punctuation,
    fe_add_speaker_avg_number_ratio=fe_add_speaker_avg_number_ratio,
    fe_add_negation_count=fe_add_negation_count, fe_add_hedge_count=fe_add_hedge_count,
    fe_add_absolutist_count=fe_add_absolutist_count, fe_add_numeral_count=fe_add_numeral_count,
    fe_add_proper_noun_count=fe_add_proper_noun_count, fe_add_readability=fe_add_readability,
    fe_add_sentiment=fe_add_sentiment, fe_scale=fe_scale, fe_verbose=fe_verbose,
)
print('OneStepOptions built.')

## Run Preprocessing

Slowest step (~5–10 min) due to sentence embedding encoding. The result is a fully numeric feature table plus string categoricals to be OrdinalEncoded.

In [ ]:
print(f'[{_now()}] Running preprocessing...')
_t0 = time()
df_processed = preprocess_one_step(df, options=options)
print(f'Done in {time()-_t0:.1f}s  |  Rows: {len(df_processed):,}  |  Columns: {df_processed.shape[1]}')
df_processed.dtypes.value_counts()

## Categorical Encoding

String columns are split into two groups:
- **Text strings** (`*_clean`, `*_original`, raw source columns): dropped — not useful as tree features.
- **Grouped categories / interaction keys** (`speaker_grouped`, `fe_speaker_subject`, `state_info_us_region`, etc.): encoded to integers with `OrdinalEncoder`.

**Why not native LightGBM categorical support?** LightGBM can handle categoricals natively if passed as `categorical_feature`, which applies Fisher's optimal split — better than treating ordinal integers as numeric. In this project, the OrdinalEncoder approach was used for consistency across RFC, LGBM, and CatBoost; CatBoost's ordered boosting provides a more rigorous alternative.

In [ ]:
_all_obj_cols = df_processed.select_dtypes(include='object').columns.tolist()
_source_cols = {
    statement_source_col, speaker_source_col, subject_source_col,
    speaker_job_source_col, party_affiliation_source_col, state_source_col,
}
_text_cols = (
    {c for c in _all_obj_cols if c.endswith(('_clean', '_original'))}
    | (_source_cols & set(_all_obj_cols))
)
_cat_cols = [c for c in _all_obj_cols if c not in _text_cols and c != label_source_col]

print(f'Text columns dropped     : {sorted(_text_cols)}')
print(f'Categorical cols encoded : {_cat_cols}')

ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
_cat_encoded = pd.DataFrame(
    ordinal_enc.fit_transform(df_processed[_cat_cols]),
    columns=_cat_cols,
    index=df_processed.index,
)

df_features = pd.concat(
    [df_processed.select_dtypes(exclude='object'), _cat_encoded],
    axis=1,
)
print(f'\nTotal features after encoding: {df_features.shape[1]}')

## Feature Matrix

In [ ]:
X = df_features.drop(columns=[label_source_col])
y = df_processed[label_source_col]

vec_cols   = [c for c in X.columns if '_vec_' in c or c.startswith('vec_')]
cat_cols_X = [c for c in X.columns if c in _cat_cols]
other_cols = [c for c in X.columns if c not in vec_cols and c not in cat_cols_X]

print(f'Embedding dimensions : {len(vec_cols)}')
print(f'Encoded categoricals : {len(cat_cols_X)}  →  {cat_cols_X}')
print(f'Other numeric        : {len(other_cols)}  →  {other_cols}')
print(f'Total features       : {X.shape[1]}')
print(f'\nTarget distribution:\n{y.value_counts(normalize=True).round(4)}')

## Train / Holdout Split

20% stratified holdout. `random_state=42` is shared across all experiments so the holdout rows are identical.

In [ ]:
X_trainval, X_holdout, y_trainval, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train/val : {X_trainval.shape[0]:,}  |  Holdout : {X_holdout.shape[0]:,}  |  CV folds : {skf.get_n_splits()}')

## Model Configuration — Option C

**LightGBM key parameters explained:**

| Parameter | Role |
|---|---|
| `num_leaves` | Max leaves per tree — primary capacity control (leaf-wise growth). Roughly equivalent to `2^max_depth` but more flexible. |
| `min_child_samples` | Min samples in a leaf — the most important regulariser. Prevents tiny over-specific leaves. |
| `subsample` | Row fraction sampled per tree — stochastic gradient boosting, reduces variance. |
| `colsample_bytree` | Column fraction sampled per tree — implicit feature selection, reduces correlation. |
| `reg_alpha` | L1 penalty on leaf weights — promotes sparse models; can zero out uninformative leaves. |
| `reg_lambda` | L2 penalty on leaf weights — shrinks leaf values toward zero; smooths predictions. |
| `verbose=-1` | Silences LightGBM's per-iteration output. |

**Option C rationale:** `drop_speaker_true_rate = False` restores the strongest signal. Instead of removing it, L1/L2 regularisation is added to the HP search to dampen the gain monopoly by penalising extreme leaf values.

In [ ]:
CLASS_WEIGHT  = {0: 1.42, 1: 0.77}
THRESHOLD     = 0.5

enable_threshold_tuning = True
overwrite_threshold     = True
THRESHOLD_METRIC        = 'macro_f1'

enable_hp_search = True
N_ITER_SEARCH    = 20
N_CV_INNER       = 3
HP_SEARCH_METRIC = 'f1_macro'

param_dist = {
    'n_estimators':      [300, 500, 800],
    'learning_rate':     [0.03, 0.05, 0.1, 0.2],
    'num_leaves':        [31, 63, 127],
    'min_child_samples': randint(10, 50),     # uniform integer 10–49
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
    'reg_alpha':         [0.0, 0.1, 0.5, 1.0],       # L1 regularisation
    'reg_lambda':        [0.0, 0.1, 0.5, 1.0, 5.0],  # L2 regularisation
}

# Option C: fe_speaker_true_rate INCLUDED; reg_alpha/reg_lambda added to dampen dominance
drop_speaker_true_rate    = False
enable_true_rate_features = True
true_rate_fallback        = 0.5

## True-Rate Feature Setup

True-rate features (mean label per speaker/subject/party/job group) are the strongest individual signals on PolitiFact data.

**Leakage prevention:** Computed only on the training split of each CV fold, then mapped onto the validation split. Placeholder `0.5` is written now; real means overwrite inside the loop.

**All four features included** (Option C): `fe_speaker_true_rate`, `fe_subject_true_rate`, `fe_party_true_rate`, `fe_speaker_job_true_rate`.

In [ ]:
_tr_group_cols: dict = {}
if enable_true_rate_features:
    _candidates = {
        'fe_speaker_true_rate':     ['speaker_grouped', 'speaker_clean'],
        'fe_subject_true_rate':     ['subject_primary_grouped', 'subject_primary', 'subject_clean'],
        'fe_party_true_rate':       ['party_affiliation_grouped', 'party_affiliation_clean'],
        'fe_speaker_job_true_rate': ['speaker_job_grouped', 'speaker_job_clean'],
    }
    if drop_speaker_true_rate:
        _candidates.pop('fe_speaker_true_rate', None)

    for _feat, _src_cols in _candidates.items():
        for _col in _src_cols:
            if _col in df_processed.columns:
                _tr_group_cols[_feat] = _col
                break

    X_trainval = X_trainval.copy()
    X_holdout  = X_holdout.copy()
    for _feat in _tr_group_cols:
        X_trainval[_feat] = true_rate_fallback
        X_holdout[_feat]  = true_rate_fallback

    _grp_trainval = pd.DataFrame(
        {col: df_processed[col].loc[X_trainval.index].values for col in _tr_group_cols.values()}
    )
    _grp_trainval['_label'] = y_trainval.values

    _grp_holdout = pd.DataFrame(
        {col: df_processed[col].loc[X_holdout.index].values for col in _tr_group_cols.values()}
    )

    print(f'True-rate features : {list(_tr_group_cols.keys())}')
    print(f'Source columns     : {list(_tr_group_cols.values())}')

## Cross-Validation

**Nested CV design:**
- **Outer loop (5 folds):** collects OOF probabilities and unbiased metric estimates.
- **Inner loop (3-fold `RandomizedSearchCV`, 20 trials):** HP search per fold with `refit=True`. `sklearn.clone()` works with LightGBM so `RandomizedSearchCV` is used directly (unlike CatBoost where we had to implement manual inner CV).
- `n_jobs=1` on the inner LGBM; `n_jobs=-1` on `RandomizedSearchCV` parallelises across the 20 parameter trials.

True-rate features are recomputed from the fold training split each time.

In [ ]:
print(f'[{_now()}] Starting cross-validation...')
_cv_start = time()
cv_fold_metrics = []
oof_proba = np.zeros(len(X_trainval))
oof_true  = np.zeros(len(X_trainval), dtype=int)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval), 1):
    _fold_t = time()

    X_fold_train = X_trainval.iloc[train_idx].copy()
    X_fold_val   = X_trainval.iloc[val_idx].copy()
    y_fold_train = y_trainval.iloc[train_idx]
    y_fold_val   = y_trainval.iloc[val_idx]

    if enable_true_rate_features and _tr_group_cols:
        _grp_tr = _grp_trainval.iloc[train_idx]
        _grp_vl = _grp_trainval.iloc[val_idx]
        for _feat, _src_col in _tr_group_cols.items():
            _rate_map = _grp_tr.groupby(_src_col)['_label'].mean()
            X_fold_train[_feat] = _grp_tr[_src_col].map(_rate_map).fillna(true_rate_fallback).values
            X_fold_val[_feat]   = _grp_vl[_src_col].map(_rate_map).fillna(true_rate_fallback).values

    if enable_hp_search:
        _base_lgbm = LGBMClassifier(
            class_weight=CLASS_WEIGHT,
            n_jobs=1,
            random_state=42,
            verbose=-1,
        )
        _inner_cv = StratifiedKFold(n_splits=N_CV_INNER, shuffle=True, random_state=42)
        _inner_search = RandomizedSearchCV(
            _base_lgbm,
            param_distributions=param_dist,
            n_iter=N_ITER_SEARCH,
            scoring=HP_SEARCH_METRIC,
            cv=_inner_cv,
            refit=True,
            random_state=42,
            n_jobs=-1,
            error_score='raise',
        )
        _inner_search.fit(X_fold_train, y_fold_train)
        fold_model       = _inner_search.best_estimator_
        fold_best_params = _inner_search.best_params_
        print(f'  Fold {fold_idx} HP best: {fold_best_params}  (inner {HP_SEARCH_METRIC}={_inner_search.best_score_:.4f})')
    else:
        fold_model = LGBMClassifier(
            n_estimators=500, learning_rate=0.1, num_leaves=63, min_child_samples=20,
            class_weight=CLASS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1,
        )
        fold_model.fit(X_fold_train, y_fold_train)
        fold_best_params = {}

    y_fold_pred  = fold_model.predict(X_fold_val)
    y_fold_proba = fold_model.predict_proba(X_fold_val)[:, 1]

    oof_proba[val_idx] = y_fold_proba
    oof_true[val_idx]  = y_fold_val.values

    fold_metrics = {
        'fold':         fold_idx,
        'roc_auc':      roc_auc_score(y_fold_val, y_fold_proba),
        'pr_auc':       average_precision_score(y_fold_val, y_fold_proba),
        'macro_f1':     f1_score(y_fold_val, y_fold_pred, average='macro', zero_division=0),
        'f1':           f1_score(y_fold_val, y_fold_pred, zero_division=0),
        'precision':    precision_score(y_fold_val, y_fold_pred, zero_division=0),
        'recall':       recall_score(y_fold_val, y_fold_pred, zero_division=0),
        'accuracy':     accuracy_score(y_fold_val, y_fold_pred),
        'mcc':          matthews_corrcoef(y_fold_val, y_fold_pred),
        'balanced_acc': balanced_accuracy_score(y_fold_val, y_fold_pred),
        'best_params':  fold_best_params,
    }
    cv_fold_metrics.append(fold_metrics)

    print(
        f'  Fold {fold_idx} | '
        f'ROC-AUC={fold_metrics["roc_auc"]:.4f}  '
        f'Macro-F1={fold_metrics["macro_f1"]:.4f}  '
        f'MCC={fold_metrics["mcc"]:.4f}  '
        f'Bal-Acc={fold_metrics["balanced_acc"]:.4f}  '
        f'({time()-_fold_t:.1f}s)'
    )

print(f'\nTotal CV time: {time()-_cv_start:.1f}s')

## CV Summary

In [ ]:
_cv_keys = ['roc_auc', 'pr_auc', 'macro_f1', 'f1', 'precision', 'recall', 'accuracy', 'mcc', 'balanced_acc']
cv_summary = {}
for k in _cv_keys:
    vals = [m[k] for m in cv_fold_metrics]
    cv_summary[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

summary_df = pd.DataFrame(cv_summary).T.round(4)
print('Cross-validation summary (5-fold):')
display(summary_df)

folds_df = pd.DataFrame([
    {k: v for k, v in m.items() if k != 'best_params'}
    for m in cv_fold_metrics
])
print('\nPer-fold results:')
display(folds_df.set_index('fold').round(4))

## HP Aggregation

Aggregation rules match the parameter types:
- **List HPs** (`n_estimators`, `learning_rate`, `num_leaves`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`): **mode** across 5 folds.
- **Integer HP** (`min_child_samples`): **median** across 5 folds (robust to outlier folds).

Observe whether `reg_alpha` or `reg_lambda` values above 0 are consistently chosen — that would confirm that regularisation helps dampen `fe_speaker_true_rate` dominance.

In [ ]:
_best_params_final = {
    'n_estimators':      500,
    'learning_rate':     0.1,
    'num_leaves':        63,
    'min_child_samples': 20,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.0,
    'reg_lambda':        0.0,
}

if enable_hp_search:
    _all_fold_params = [m['best_params'] for m in cv_fold_metrics if m.get('best_params')]

    for _hp in ['n_estimators', 'learning_rate', 'num_leaves', 'subsample',
                'colsample_bytree', 'reg_alpha', 'reg_lambda']:
        _vals = [p[_hp] for p in _all_fold_params if _hp in p]
        if _vals:
            _counts = Counter(_vals).most_common()
            _mode   = _counts[0][0]
            _best_params_final[_hp] = _mode
            print(f'  {_hp:25s}: votes={_counts}  → chosen: {_mode}')

    for _hp in ['min_child_samples']:
        _vals = [p[_hp] for p in _all_fold_params if _hp in p]
        if _vals:
            _median = int(np.median(_vals))
            _best_params_final[_hp] = _median
            print(f'  {_hp:25s}: values={sorted(_vals)}  → median: {_median}')

print(f'\nFinal HP for fit: {_best_params_final}')

## Threshold Tuning

Default 0.5 favours the majority class. Sweeping thresholds from 0.20 to 0.76 on OOF probabilities and picking the one that maximises macro F1 — no retraining required.

In [ ]:
if enable_threshold_tuning:
    _metric_fn = {
        'macro_f1':     lambda t, yt, yp: f1_score(yt, yp, average='macro', zero_division=0),
        'mcc':          lambda t, yt, yp: matthews_corrcoef(yt, yp),
        'balanced_acc': lambda t, yt, yp: balanced_accuracy_score(yt, yp),
    }[THRESHOLD_METRIC]

    threshold_grid   = np.arange(0.20, 0.76, 0.02)
    threshold_scores = {}
    for t in threshold_grid:
        preds = (oof_proba >= t).astype(int)
        threshold_scores[round(float(t), 2)] = _metric_fn(t, oof_true, preds)

    best_threshold = max(threshold_scores, key=threshold_scores.get)
    best_score     = threshold_scores[best_threshold]

    fig, ax = plt.subplots(figsize=(9, 4))
    ts = list(threshold_scores.keys())
    ss = list(threshold_scores.values())
    ax.plot(ts, ss, marker='o', markersize=4, linewidth=1.5, label=THRESHOLD_METRIC)
    ax.axvline(best_threshold, color='red', linestyle='--', alpha=0.7,
               label=f'Best = {best_threshold:.2f} ({best_score:.4f})')
    ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='Default = 0.50')
    ax.set_xlabel('Threshold')
    ax.set_ylabel(THRESHOLD_METRIC)
    ax.set_title('Threshold Sweep on OOF Probabilities')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Best threshold: {best_threshold:.2f}  (OOF {THRESHOLD_METRIC}={best_score:.4f})')

    if overwrite_threshold:
        print(f'THRESHOLD updated: {THRESHOLD:.2f} → {best_threshold:.2f}')
        THRESHOLD = best_threshold

## Final Model — Fit on Full Train/Val Set

In [ ]:
print(f'[{_now()}] Fitting final model...')
_t0 = time()

_final_rate_maps: dict = {}
X_trainval_final = X_trainval.copy()
if enable_true_rate_features and _tr_group_cols:
    for _feat, _src_col in _tr_group_cols.items():
        _rate_map = _grp_trainval.groupby(_src_col)['_label'].mean()
        X_trainval_final[_feat] = _grp_trainval[_src_col].map(_rate_map).fillna(true_rate_fallback).values
        X_holdout[_feat]        = _grp_holdout[_src_col].map(_rate_map).fillna(true_rate_fallback).values
        _final_rate_maps[_feat] = _rate_map.to_dict()

model = LGBMClassifier(
    n_estimators=_best_params_final['n_estimators'],
    learning_rate=_best_params_final['learning_rate'],
    num_leaves=_best_params_final['num_leaves'],
    min_child_samples=_best_params_final['min_child_samples'],
    subsample=_best_params_final['subsample'],
    colsample_bytree=_best_params_final['colsample_bytree'],
    reg_alpha=_best_params_final['reg_alpha'],
    reg_lambda=_best_params_final['reg_lambda'],
    class_weight=CLASS_WEIGHT,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
model.fit(X_trainval_final, y_trainval)
print(f'Done in {time()-_t0:.1f}s')

## Holdout Evaluation

The holdout set is touched for the first time here.

In [ ]:
print(f'Using threshold: {THRESHOLD:.2f}\n')
y_proba = model.predict_proba(X_holdout)[:, 1]
y_pred  = (y_proba >= THRESHOLD).astype(int)

holdout_metrics = {
    'roc_auc':      roc_auc_score(y_holdout, y_proba),
    'pr_auc':       average_precision_score(y_holdout, y_proba),
    'macro_f1':     f1_score(y_holdout, y_pred, average='macro', zero_division=0),
    'f1_class1':    f1_score(y_holdout, y_pred, zero_division=0),
    'precision':    precision_score(y_holdout, y_pred, zero_division=0),
    'recall':       recall_score(y_holdout, y_pred, zero_division=0),
    'accuracy':     accuracy_score(y_holdout, y_pred),
    'mcc':          matthews_corrcoef(y_holdout, y_pred),
    'balanced_acc': balanced_accuracy_score(y_holdout, y_pred),
}
cm     = confusion_matrix(y_holdout, y_pred)
report = classification_report(y_holdout, y_pred, output_dict=True)

metrics_df = pd.DataFrame.from_dict(holdout_metrics, orient='index', columns=['value']).round(4)
display(metrics_df)
print()
print(classification_report(y_holdout, y_pred, target_names=['True (0)', 'False (1)']))

## Feature Importance — Gain

LightGBM provides two importance types:
- **split**: number of times a feature is used in a split. Fast but biased toward high-cardinality features.
- **gain**: total reduction in loss (impurity) attributed to each feature across all splits. More informative — this is what we use.

Accessed via `model.booster_.feature_importance(importance_type='gain')`. This is the importance that revealed the 6–8× dominance of `fe_speaker_true_rate` in the original Option A experiment.

In [ ]:
feature_names_final = X_trainval_final.columns.tolist()
importances = model.booster_.feature_importance(importance_type='gain')
importance_df = (
    pd.DataFrame({'feature': feature_names_final, 'importance': importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print('Top 30 features (gain importance):')
display(importance_df.head(30))

# Dominance ratio: top feature vs mean of remaining top-10
top_gain      = importance_df['importance'].iloc[0]
second_gains  = importance_df['importance'].iloc[1:10].mean()
print(f'\nDominance ratio (top / mean of 2nd–10th): {top_gain / second_gains:.1f}×')
print(f'Top feature: {importance_df["feature"].iloc[0]}')

## Evaluation Plots

Four panels:
1. **ROC curve** — threshold-independent ranking ability.
2. **Precision-Recall curve** — better suited for class-imbalanced problems.
3. **Confusion matrix** — per-cell counts at the tuned threshold.
4. **Feature importance** — top 20 features by gain.

In [ ]:
fpr, tpr, _      = roc_curve(y_holdout, y_proba)
prec_c, rec_c, _ = precision_recall_curve(y_holdout, y_proba)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# ROC
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {holdout_metrics['roc_auc']:.4f}")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.6)
axes[0].set_title('ROC Curve (holdout)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR
axes[1].plot(rec_c, prec_c, label=f"PR-AUC = {holdout_metrics['pr_auc']:.4f}")
axes[1].set_title('Precision-Recall Curve (holdout)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confusion matrix
im = axes[2].imshow(cm, interpolation='nearest', cmap='Blues')
axes[2].set_title(f'Confusion Matrix (threshold={THRESHOLD:.2f})')
axes[2].set_xticks([0, 1])
axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['Pred: True (0)', 'Pred: False (1)'])
axes[2].set_yticklabels(['Actual: True (0)', 'Actual: False (1)'])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
fig.colorbar(im, ax=axes[2])

# Feature importance (top 20)
top20 = importance_df.head(20)
axes[3].barh(top20['feature'].values[::-1], top20['importance'].values[::-1], color='steelblue')
axes[3].set_title('Top 20 Feature Importances (Gain)')
axes[3].set_xlabel('Gain')
axes[3].tick_params(axis='y', labelsize=7)
axes[3].grid(True, alpha=0.3, axis='x')

plt.suptitle(
    f'LightGBM Option C — Holdout  |  macro_f1={holdout_metrics["macro_f1"]:.4f}  '
    f'ROC-AUC={holdout_metrics["roc_auc"]:.4f}  threshold={THRESHOLD:.2f}',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

## Feature Importance — Full Bar Chart (Top 30, colour-coded)

Colour legend:
- **Red**: embedding dimensions (`vec_*`) — semantic content
- **Gold**: true-rate features (`*_true_rate`) — historical false-claim rates per group
- **Blue**: metadata and text-style features

With Option C (speaker true-rate restored + regularisation), watch whether `fe_speaker_true_rate` still dominates or whether regularisation has spread the gain more evenly.

In [ ]:
top30 = importance_df.head(30)
colors = [
    'tomato'    if ('_vec_' in f or f.startswith('vec_')) else
    'goldenrod' if '_true_rate' in f else
    'steelblue'
    for f in top30['feature'].values[::-1]
]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top30['feature'].values[::-1], top30['importance'].values[::-1], color=colors)
ax.set_title('Top 30 Feature Importances (Gain) — red=embedding, gold=true-rate, blue=other')
ax.set_xlabel('Gain')
ax.tick_params(axis='y', labelsize=8)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Summary

| Metric | CV (OOF mean) | Holdout |
|---|---|---|
| Macro F1 | (see above) | (see above) |
| ROC-AUC | (see above) | (see above) |
| MCC | (see above) | (see above) |
| Balanced Accuracy | (see above) | (see above) |

**Option A → B → C progression:**

| Option | `fe_speaker_true_rate` | Regularisation | Key finding |
|---|---|---|---|
| A | included | none | Speaker true-rate dominates at 6–8× gain ratio |
| B | dropped | none | Other features claim more gain; holdout F1 may improve |
| C | included | `reg_alpha`, `reg_lambda` | Tests whether L1/L2 can reduce dominance without removing the signal |

**Key LightGBM vs RFC takeaways:**
- Gain importance is more informative than RFC's MDI — it directly quantifies how much each feature contributes to reducing the loss, rather than counting splits.
- Leaf-wise growth makes LGBM faster and often more accurate than RFC on tabular data.
- The dominance discovery motivated the move to CatBoost (ordered boosting) and eventually to transformer-only models that bypass structured features altogether.